# Gradient Descent — Exercises

8 exercises covering gradient descent convergence, momentum, Nesterov acceleration, line search, and implicit bias.

| Format            | Description                                  |
| ----------------- | -------------------------------------------- |
| **Problem**       | Markdown cell with task description          |
| **Your Solution** | Code cell with scaffolding                   |
| **Solution**      | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus                |
| ----- | --------- | -------------------- |
| ★     | 1-3       | Core mechanics       |
| ★★    | 4-6       | Deeper theory        |
| ★★★   | 7-8       | AI / ML applications |

### Topic Map

| Topic    | Exercise |
| -------- | -------- |
| GD on quadratics | 1, 2 |
| Condition number effects | 3 |
| Momentum analysis | 4 |
| Nesterov vs momentum | 5 |
| Armijo backtracking | 6 |
| Implicit bias | 7 |
| Neural network training | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', expected)
        print('  got     :', got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def gd_step(x, grad, eta):
    return x - eta * grad

def momentum_step(x, v, grad, eta, beta):
    v_new = beta * v + grad
    x_new = x - eta * v_new
    return x_new, v_new

def nag_step(x, y, x_prev, t, grad_fn, eta):
    x_new = y - eta * grad_fn(y)
    t_new = (1 + np.sqrt(1 + 4*t**2)) / 2
    y_new = x_new + ((t - 1) / t_new) * (x_new - x_prev)
    return x_new, y_new, t_new

print('Setup complete.')

---

### Exercise 1: GD on a 2D Quadratic ★

Implement gradient descent on $f(\mathbf{x}) = \frac{1}{2}(x_1^2 + \kappa x_2^2)$ and verify convergence.

**(a)** Compute the gradient $\nabla f(\mathbf{x})$.

**(b)** Run GD with $\eta = 1/L$ where $L = \kappa$, starting from $\mathbf{x}_0 = (1, 1)$.

**(c)** Verify that after 50 steps, $f(\mathbf{x}_{50}) < 0.01$ for $\kappa = 10$.

**(d)** Compute the theoretical bound $\frac{L \lVert \mathbf{x}_0 \rVert^2}{2T}$ and verify the error is below it.

In [ ]:
# Exercise 1: Your Solution
kappa = 10
A = np.diag([1.0, kappa])
L = kappa
eta = 1.0 / L
x0 = np.array([1.0, 1.0])
T = 50

def f_quad(x, A):
    # YOUR CODE HERE
    pass

def grad_quad(x, A):
    # YOUR CODE HERE
    pass

x = x0.copy()
for t in range(T):
    g = grad_quad(x, A)
    x = gd_step(x, g, eta)

final_error = f_quad(x, A)
print(f'f(x_{T}) = {final_error:.8f}')

In [ ]:
# Exercise 1: Solution
kappa = 10
A = np.diag([1.0, kappa])
L = kappa
eta = 1.0 / L
x0 = np.array([1.0, 1.0])
T = 50

def f_quad(x, A):
    return 0.5 * x @ A @ x

def grad_quad(x, A):
    return A @ x

x = x0.copy()
for t in range(T):
    g = grad_quad(x, A)
    x = gd_step(x, g, eta)

header('Exercise 1: GD on a 2D Quadratic')
final_error = f_quad(x, A)
print(f'f(x_{T}) = {final_error:.8f}')

check_close('error < 0.01', final_error < 0.01, True)

bound = L * np.linalg.norm(x0)**2 / (2 * T)
print(f'Theoretical bound: {bound:.8f}')
check_true('error below theoretical bound', final_error <= bound)

print('\nTakeaway: GD on quadratics converges at rate O(1/T), with the constant controlled by L and the initial distance to optimum.')

---

### Exercise 2: Convergence Rate Verification ★

Verify the $O(1/T)$ convergence rate by running GD with different step sizes.

**(a)** Run GD on $f(x) = x^2$ with $\eta \in \{0.1, 0.5, 1.0, 1.5, 2.0\}$ starting from $x_0 = 10$.

**(b)** For each $\eta$, record whether the algorithm converges or diverges after 100 steps.

**(c)** Identify the stability boundary (the largest $\eta$ for which convergence occurs).

**(d)** Verify that the stability boundary is $2/L$ where $L = f''(x) = 2$.

In [ ]:
# Exercise 2: Your Solution
def f_1d(x): return x**2
def grad_1d(x): return 2*x
L_1d = 2.0
x0 = 10.0
T = 100

etas = [0.1, 0.5, 1.0, 1.5, 2.0]
results = {}

for eta in etas:
    x = x0
    diverged = False
    for t in range(T):
        x = x - eta * grad_1d(x)
        if abs(x) > 1e10:
            diverged = True
            break
    results[eta] = ('diverged' if diverged else f'converged, x={x:.6f}')

for eta, status in results.items():
    print(f'eta={eta}: {status}')

In [ ]:
# Exercise 2: Solution
def f_1d(x): return x**2
def grad_1d(x): return 2*x
L_1d = 2.0
x0 = 10.0
T = 100

etas = [0.1, 0.5, 1.0, 1.5, 2.0]
results = {}

for eta in etas:
    x = x0
    diverged = False
    for t in range(T):
        x = x - eta * grad_1d(x)
        if abs(x) > 1e10:
            diverged = True
            break
    results[eta] = ('diverged' if diverged else f'converged, x={x:.6f}')

header('Exercise 2: Convergence Rate Verification')
for eta, status in results.items():
    print(f'eta={eta}: {status}')

print(f'\nStability boundary: 2/L = {2/L_1d}')
converged_etas = [e for e, s in results.items() if 'converged' in s]
diverged_etas = [e for e, s in results.items() if 'diverged' in s]
boundary_ok = max(converged_etas) < 2/L_1d <= min(diverged_etas) if diverged_etas else True
check_true('stability boundary is 2/L', boundary_ok)

print('\nTakeaway: GD is stable only when eta < 2/L. For f(x)=x^2, L=2, so the boundary is eta=1.0.')

---

### Exercise 3: Condition Number and Convergence ★

Explore how the condition number affects GD convergence speed.

**(a)** Construct 2D quadratics with $\kappa \in \{1, 10, 100, 1000\}$.

**(b)** For each, run GD with $\eta = 1/L$ from $\mathbf{x}_0 = (1, 1)$ and count iterations to reach $f(\mathbf{x}) < 10^{-6}$.

**(c)** Verify that iterations scale approximately as $O(\kappa)$.

**(d)** Compute the ratio of iterations for $\kappa=100$ vs $\kappa=10$.

In [ ]:
# Exercise 3: Your Solution
kappa_values = [1, 10, 100, 1000]
x0 = np.array([1.0, 1.0])
target = 1e-6
max_iters = 100000

iters_needed = []
for kappa in kappa_values:
    A = np.diag([1.0, kappa])
    L = kappa
    eta = 1.0 / L
    x = x0.copy()
    for t in range(max_iters):
        x = x - eta * (A @ x)
        if 0.5 * x @ A @ x < target:
            iters_needed.append(t + 1)
            break
    else:
        iters_needed.append(-1)

for k, iters in zip(kappa_values, iters_needed):
    print(f'kappa={k:<5} -> {iters} iterations')

In [ ]:
# Exercise 3: Solution
kappa_values = [1, 10, 100, 1000]
x0 = np.array([1.0, 1.0])
target = 1e-6
max_iters = 100000

iters_needed = []
for kappa in kappa_values:
    A = np.diag([1.0, kappa])
    L = kappa
    eta = 1.0 / L
    x = x0.copy()
    for t in range(max_iters):
        x = x - eta * (A @ x)
        if 0.5 * x @ A @ x < target:
            iters_needed.append(t + 1)
            break
    else:
        iters_needed.append(-1)

header('Exercise 3: Condition Number Effects')
for k, iters in zip(kappa_values, iters_needed):
    print(f'kappa={k:<5} -> {iters} iterations')

# Verify O(kappa) scaling
if iters_needed[1] > 0 and iters_needed[2] > 0:
    ratio = iters_needed[2] / iters_needed[1]
    print(f'\nRatio iters(100)/iters(10) = {ratio:.1f} (expected ~10)')
    check_close('ratio approximately 10', ratio, 10.0, tol=3.0)

print('\nTakeaway: GD iterations scale linearly with condition number kappa. Ill-conditioned problems are slow.')


---

### Exercise 4: Momentum Analysis ★★

Analyze how momentum accelerates convergence on ill-conditioned problems.

**(a)** For $f(\mathbf{x}) = \frac{1}{2}(x_1^2 + 100 x_2^2)$, run vanilla GD and momentum ($\beta = 0.9$) for 200 iterations.

**(b)** Compute the optimal momentum parameter $\beta^* = \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2$.

**(c)** Run GD with momentum using $\beta^*$ and compare all three methods.

**(d)** Compute the speedup factor: error of vanilla GD / error of optimal momentum.

In [ ]:
# Exercise 4: Your Solution
kappa = 100
A = np.diag([1.0, kappa])
L = kappa
eta = 1.0 / L
x0 = np.array([1.0, 1.0])
T = 200

# Vanilla GD
x = x0.copy()
for _ in range(T):
    x = x - eta * (A @ x)
err_gd = 0.5 * x @ A @ x

# Momentum with beta=0.9
beta_default = 0.9
x = x0.copy()
v = np.zeros(2)
for _ in range(T):
    v = beta_default * v + A @ x
    x = x - eta * v
err_mom_default = 0.5 * x @ A @ x

# Optimal momentum
beta_opt = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1))**2
x = x0.copy()
v = np.zeros(2)
for _ in range(T):
    v = beta_opt * v + A @ x
    x = x - eta * v
err_mom_opt = 0.5 * x @ A @ x

print(f'GD error: {err_gd:.2e}')
print(f'Momentum (0.9) error: {err_mom_default:.2e}')
print(f'Momentum (opt={beta_opt:.3f}) error: {err_mom_opt:.2e}')

In [ ]:
# Exercise 4: Solution
kappa = 100
A = np.diag([1.0, kappa])
L = kappa
eta = 1.0 / L
x0 = np.array([1.0, 1.0])
T = 200

x = x0.copy()
for _ in range(T):
    x = x - eta * (A @ x)
err_gd = 0.5 * x @ A @ x

beta_default = 0.9
x = x0.copy()
v = np.zeros(2)
for _ in range(T):
    v = beta_default * v + A @ x
    x = x - eta * v
err_mom_default = 0.5 * x @ A @ x

beta_opt = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1))**2
x = x0.copy()
v = np.zeros(2)
for _ in range(T):
    v = beta_opt * v + A @ x
    x = x - eta * v
err_mom_opt = 0.5 * x @ A @ x

header('Exercise 4: Momentum Analysis')
print(f'GD error: {err_gd:.2e}')
print(f'Momentum (0.9) error: {err_mom_default:.2e}')
print(f'Momentum (opt={beta_opt:.3f}) error: {err_mom_opt:.2e}')

speedup = err_gd / max(err_mom_opt, 1e-30)
print(f'\nSpeedup: {speedup:.1f}x')
check_true('momentum outperforms GD', err_mom_opt < err_gd)
check_true('optimal beta beats default', err_mom_opt <= err_mom_default)

print('\nTakeaway: Momentum provides dramatic speedup on ill-conditioned problems. The optimal beta depends on kappa.')

---

### Exercise 5: Nesterov vs. Polyak Momentum ★★

Compare Nesterov accelerated gradient with Polyak momentum on a convex function.

**(a)** Implement NAG for $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top A \mathbf{x}$ with $A = \text{diag}(1, 100)$.

**(b)** Run both NAG and Polyak momentum for 300 iterations from $\mathbf{x}_0 = (1, 1)$.

**(c)** Compare final errors and verify NAG achieves a better rate.

**(d)** Plot the convergence curves (if matplotlib is available).

In [ ]:
# Exercise 5: Your Solution
A = np.diag([1.0, 100.0])
L = 100.0
eta = 1.0 / L
x0 = np.array([1.0, 1.0])
T = 300

# Polyak momentum
beta = ((np.sqrt(100) - 1) / (np.sqrt(100) + 1))**2
x = x0.copy()
v = np.zeros(2)
err_mom_hist = []
for _ in range(T):
    v = beta * v + A @ x
    x = x - eta * v
    err_mom_hist.append(0.5 * x @ A @ x)

# NAG
y = x0.copy()
x_prev = x0.copy()
t_val = 1.0
err_nag_hist = []
for _ in range(T):
    x_new = y - eta * (A @ y)
    t_new = (1 + np.sqrt(1 + 4*t_val**2)) / 2
    y = x_new + ((t_val - 1) / t_new) * (x_new - x_prev)
    x_prev = x_new
    t_val = t_new
    err_nag_hist.append(0.5 * x_new @ A @ x_new)

print(f'Momentum final error: {err_mom_hist[-1]:.2e}')
print(f'NAG final error: {err_nag_hist[-1]:.2e}')

In [ ]:
# Exercise 5: Solution
A = np.diag([1.0, 100.0])
L = 100.0
eta = 1.0 / L
x0 = np.array([1.0, 1.0])
T = 300

beta = ((np.sqrt(100) - 1) / (np.sqrt(100) + 1))**2
x = x0.copy()
v = np.zeros(2)
err_mom_hist = []
for _ in range(T):
    v = beta * v + A @ x
    x = x - eta * v
    err_mom_hist.append(0.5 * x @ A @ x)

y = x0.copy()
x_prev = x0.copy()
t_val = 1.0
err_nag_hist = []
for _ in range(T):
    x_new = y - eta * (A @ y)
    t_new = (1 + np.sqrt(1 + 4*t_val**2)) / 2
    y = x_new + ((t_val - 1) / t_new) * (x_new - x_prev)
    x_prev = x_new
    t_val = t_new
    err_nag_hist.append(0.5 * x_new @ A @ x_new)

header('Exercise 5: Nesterov vs. Polyak Momentum')
print(f'Momentum final error: {err_mom_hist[-1]:.2e}')
print(f'NAG final error: {err_nag_hist[-1]:.2e}')

check_true('NAG outperforms momentum', err_nag_hist[-1] <= err_mom_hist[-1])

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    ax.plot(steps, err_mom_hist, label='Momentum')
    ax.plot(steps, err_nag_hist, label='NAG')
    ax.set_yscale('log')
    ax.set_title('NAG vs Momentum convergence')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Suboptimality')
    ax.legend()
    fig.tight_layout()
    plt.show()

print('\nTakeaway: NAG achieves O(1/T^2) vs momentums O(1/T), giving quadratic improvement for convex functions.')

---

### Exercise 6: Armijo Backtracking Implementation ★★

Implement and analyze the Armijo backtracking line search.

**(a)** Implement the Armijo backtracking algorithm with parameters $\rho = 0.5$, $c = 10^{-4}$.

**(b)** Apply it to $f(\mathbf{x}) = \frac{1}{2}(x_1^2 + 100x_2^2)$ starting from $(1, 1)$ with initial step $\bar{\eta} = 1.0$.

**(c)** Record the step size chosen at each iteration and verify convergence.

**(d)** Compare the number of backtracking iterations needed at each step.

In [ ]:
# Exercise 6: Your Solution
A = np.diag([1.0, 100.0])

def f(x): return 0.5 * x @ A @ x
def grad_f(x): return A @ x

def armijo_backtrack(x, g, f, eta_bar=1.0, rho=0.5, c=1e-4, max_iter=50):
    # YOUR CODE HERE
    pass

x = np.array([1.0, 1.0])
eta_history = []
backtrack_iters = []

for step in range(20):
    g = grad_f(x)
    eta, n_back = armijo_backtrack(x, g, f)
    eta_history.append(eta)
    backtrack_iters.append(n_back)
    x = x - eta * g

print(f'Final error: {f(x):.2e}')
print(f'Step sizes: {[f"{e:.4f}" for e in eta_history[:5]]}...')

In [ ]:
# Exercise 6: Solution
A = np.diag([1.0, 100.0])

def f(x): return 0.5 * x @ A @ x
def grad_f(x): return A @ x

def armijo_backtrack(x, g, f, eta_bar=1.0, rho=0.5, c=1e-4, max_iter=50):
    eta = eta_bar
    f_x = f(x)
    slope = -np.dot(g, g)
    n_back = 0
    for _ in range(max_iter):
        if f(x + eta * (-g)) <= f_x + c * eta * slope:
            return eta, n_back
        eta *= rho
        n_back += 1
    return eta, n_back

x = np.array([1.0, 1.0])
eta_history = []
backtrack_iters = []

for step in range(20):
    g = grad_f(x)
    eta, n_back = armijo_backtrack(x, g, f)
    eta_history.append(eta)
    backtrack_iters.append(n_back)
    x = x - eta * g

header('Exercise 6: Armijo Backtracking')
print(f'Final error: {f(x):.2e}')
check_close('converged to near-zero', f(x), 0.0, tol=1e-6)

print(f'\nAverage backtracking iterations: {np.mean(backtrack_iters):.1f}')
print(f'Max backtracking iterations: {max(backtrack_iters)}')
check_true('backtracking is efficient', np.mean(backtrack_iters) < 5)

print('\nTakeaway: Armijo backtracking automatically finds good step sizes with very few function evaluations.')

---

### Exercise 7: GD Implicit Bias in Overparameterized Regression ★★★

Demonstrate that GD converges to the minimum-norm solution in overparameterized settings.

**(a)** Generate an underdetermined system with $n = 30$ equations and $d = 200$ unknowns.

**(b)** Solve it with GD starting from $\mathbf{w}_0 = \mathbf{0}$.

**(c)** Compare with the minimum-norm solution from the pseudoinverse: $\mathbf{w}^* = X^\dagger \mathbf{y}$.

**(d)** Verify $\lVert \mathbf{w}_{GD} - \mathbf{w}^* \rVert < 10^{-4}$.

In [ ]:
# Exercise 7: Your Solution
np.random.seed(42)
n, d = 30, 200
X = np.random.randn(n, d)
y = np.random.randn(n)

# Minimum norm solution
w_star = X.T @ la.solve(X @ X.T, y)

# GD from zero
L = la.norm(X.T @ X, 2)
eta = 1.0 / L
w = np.zeros(d)
T = 10000

for _ in range(T):
    grad = X.T @ (X @ w - y)
    w = w - eta * grad

print(f'||w_gd - w_star|| = {la.norm(w - w_star):.2e}')
print(f'||w_gd|| = {la.norm(w):.6f}')
print(f'||w_star|| = {la.norm(w_star):.6f}')

In [ ]:
# Exercise 7: Solution
np.random.seed(42)
n, d = 30, 200
X = np.random.randn(n, d)
y = np.random.randn(n)

w_star = X.T @ la.solve(X @ X.T, y)

L = la.norm(X.T @ X, 2)
eta = 1.0 / L
w = np.zeros(d)
T = 10000

for _ in range(T):
    grad = X.T @ (X @ w - y)
    w = w - eta * grad

header('Exercise 7: GD Implicit Bias')
print(f'||w_gd - w_star|| = {la.norm(w - w_star):.2e}')
print(f'||w_gd|| = {la.norm(w):.6f}')
print(f'||w_star|| = {la.norm(w_star):.6f}')

check_close('GD matches min-norm solution', w, w_star, tol=1e-4)
check_true('residual is near zero', la.norm(X @ w - y) < 1e-8)

print('\nTakeaway: GD from zero initialization implicitly regularizes by finding the minimum-norm solution.')


---

### Exercise 8: Learning Rate Selection for Neural Network Training ★★★

Train a simple neural network with different learning rates and analyze the results.

**(a)** Create a synthetic dataset: $y = \sin(2\pi x) + \epsilon$ with $x \in [0, 1]$.

**(b)** Train a 2-layer neural network with GD using learning rates $\eta \in \{0.001, 0.01, 0.1, 1.0\}$.

**(c)** For each LR, record the final training loss and whether training was stable.

**(d)** Identify the optimal learning rate and explain why other choices fail.

In [ ]:
# Exercise 8: Your Solution
np.random.seed(42)
n_data = 200
X_data = np.random.rand(n_data, 1)
y_data = np.sin(2 * np.pi * X_data[:, 0]) + 0.1 * np.random.randn(n_data)

# Simple 2-layer network: input -> 20 hidden (ReLU) -> output
d_in, d_hidden, d_out = 1, 20, 1

def init_params():
    W1 = np.random.randn(d_in, d_hidden) * 0.5
    b1 = np.zeros(d_hidden)
    W2 = np.random.randn(d_hidden, d_out) * 0.5
    b2 = np.zeros(d_out)
    return W1, b1, W2, b2

def relu(z): return np.maximum(0, z)
def relu_grad(z): return (z > 0).astype(float)

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    out = a1 @ W2 + b2
    return z1, a1, out

def compute_loss(out, y):
    return 0.5 * np.mean((out.flatten() - y)**2)

def train_network(eta, T=1000):
    W1, b1, W2, b2 = init_params()
    losses = []
    stable = True
    for _ in range(T):
        z1, a1, out = forward(X_data, W1, b1, W2, b2)
        loss = compute_loss(out, y_data)
        if loss > 1e6:
            stable = False
            break
        losses.append(loss)
        # Backward pass
        dout = (out.flatten() - y_data).reshape(-1, 1) / n_data
        dW2 = a1.T @ dout
        db2 = dout.sum(axis=0)
        da1 = dout @ W2.T
        dz1 = da1 * relu_grad(z1)
        dW1 = X_data.T @ dz1
        db1 = dz1.sum(axis=0)
        # Update
        W1 -= eta * dW1
        b1 -= eta * db1
        W2 -= eta * dW2
        b2 -= eta * db2
    return losses, stable

etas = [0.001, 0.01, 0.1, 1.0]
for eta in etas:
    losses, stable = train_network(eta)
    print(f'eta={eta}: stable={stable}, final_loss={losses[-1]:.6f if losses else "N/A"}')

In [ ]:
# Exercise 8: Solution
np.random.seed(42)
n_data = 200
X_data = np.random.rand(n_data, 1)
y_data = np.sin(2 * np.pi * X_data[:, 0]) + 0.1 * np.random.randn(n_data)

d_in, d_hidden, d_out = 1, 20, 1

def relu(z): return np.maximum(0, z)
def relu_grad(z): return (z > 0).astype(float)

def init_params():
    np.random.seed(42)
    W1 = np.random.randn(d_in, d_hidden) * 0.5
    b1 = np.zeros(d_hidden)
    W2 = np.random.randn(d_hidden, d_out) * 0.5
    b2 = np.zeros(d_out)
    return W1, b1, W2, b2

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    out = a1 @ W2 + b2
    return z1, a1, out

def compute_loss(out, y):
    return 0.5 * np.mean((out.flatten() - y)**2)

def train_network(eta, T=1000):
    W1, b1, W2, b2 = init_params()
    losses = []
    stable = True
    for _ in range(T):
        z1, a1, out = forward(X_data, W1, b1, W2, b2)
        loss = compute_loss(out, y_data)
        if loss > 1e6:
            stable = False
            break
        losses.append(loss)
        dout = (out.flatten() - y_data).reshape(-1, 1) / n_data
        dW2 = a1.T @ dout
        db2 = dout.sum(axis=0)
        da1 = dout @ W2.T
        dz1 = da1 * relu_grad(z1)
        dW1 = X_data.T @ dz1
        db1 = dz1.sum(axis=0)
        W1 -= eta * dW1
        b1 -= eta * db1
        W2 -= eta * dW2
        b2 -= eta * db2
    return losses, stable

etas = [0.001, 0.01, 0.1, 1.0]
all_results = {}

header('Exercise 8: Neural Network Training')
for eta in etas:
    losses, stable = train_network(eta)
    all_results[eta] = (losses, stable)
    loss_str = f'{losses[-1]:.6f}' if losses else 'N/A (diverged)'
    print(f'eta={eta}: stable={stable}, final_loss={loss_str}')

# Find best LR
stable_results = {e: (l, s) for e, (l, s) in all_results.items() if s and l}
if stable_results:
    best_eta = min(stable_results, key=lambda e: stable_results[e][0][-1])
    print(f'\nBest learning rate: {best_eta}')
    check_true('at least one LR is stable', len(stable_results) > 0)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    for eta, (losses, stable) in all_results.items():
        if losses:
            ax.plot(losses, label=f'eta={eta}' + (' (stable)' if stable else ' (diverged)'))
    ax.set_yscale('log')
    ax.set_title('Training loss for different learning rates')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    fig.tight_layout()
    plt.show()

print('\nTakeaway: LR selection is critical. Too small = slow; too large = divergence. The optimal LR balances speed and stability.')

---

## What to Review After Finishing

- [ ] Exercise 1: GD update rule and convergence on quadratics
- [ ] Exercise 2: Stability boundary eta < 2/L
- [ ] Exercise 3: Condition number controls convergence speed
- [ ] Exercise 4: Momentum accelerates ill-conditioned problems
- [ ] Exercise 5: NAG achieves O(1/T^2) vs momentum's O(1/T)
- [ ] Exercise 6: Armijo backtracking for automatic step size selection
- [ ] Exercise 7: GD implicit bias toward minimum-norm solutions
- [ ] Exercise 8: Learning rate selection in neural network training

## References

1. Nesterov, Y. (2013). *Introductory Lectures on Convex Optimization*. Springer.
2. Boyd, S. & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.
3. Sutskever, I. et al. (2013). "On the importance of initialization and momentum in deep learning." ICML.
4. Polyak, B. (1964). "Some methods of speeding up the convergence of iteration methods." USSR Computational Mathematics.